In [ ]:
import os
import pandas as pd
import gradio as gr
from datetime import datetime
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from dotenv import load_dotenv

#os.environ["OPENAI_API_KEY"] = ""
load_dotenv()




c:\Users\xlimit\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 상품 데이터
products_data = [
    {"상품ID":"P001", "상품명": "프리미엄 강아지 간식", "카테고리": "간식", "가격": 32417, "제조사": "펫스토리", "태그": "간식,강아지,프리미엄", "판매량":890},
    {"상품ID":"P002", "상품명": "프리미엄 고양이 장난감", "카테고리": "장난감", "가격": 25000, "제조사": "펫스토리", "태그": "장난감,고양이,프리미엄", "판매량":500},
    {"상품ID":"P003", "상품명": "스크래처", "카테고리": "가구", "가격": 35900, "제조사": "캣톰", "태그": "스크래처,고양이,발톱", "판매량":189},
    {"상품ID":"P004", "상품명": "강아지 목줄", "카테고리": "악세사리", "가격": 15000, "제조사": "펫스토리", "태그": "목줄,강아지,악세사리", "판매량":300},
    {"상품ID":"P005", "상품명": "고양이 캣타워", "카테고리": "가구", "가격": 120000, "제조사": "캣톰", "태그": "캣타워,고양이,가구", "판매량":150},
    {"상품ID":"P006", "상품명": "강아지 샴푸", "카테고리": "미용", "가격": 20000, "제조사": "펫스토리", "태그": "샴푸,강아지,미용", "판매량":400},
    {"상품ID":"P007", "상품명": "고양이 사료", "카테고리": "사료", "가격": 45000, "제조사": "캣톰", "태그": "사료,고양이,영양", "판매량":600},
    {"상품ID":"P008", "상품명": "강아지 장난감", "카테고리": "장난감", "가격": 18000, "제조사": "펫스토리", "태그": "장난감,강아지,놀이", "판매량":350},
]

# 고객 데이터
customers_data = [
    {"고객ID": "C001", "고객명": "김철수", "이메일": "kimchulsoo@example.com", "전화번호": "010-1234-5678"},
    {"고객ID": "C002", "고객명": "이영희", "이메일": "leeyounghee@example.com", "전화번호": "010-2345-6789"},
    {"고객ID": "C003", "고객명": "박민수", "이메일": "parkminsoo@example.com", "전화번호": "010-3456-7890"},
]

# 주문 데이터
orders_data = [
    {"주문ID": "O001", "고객ID": "C001", "상품ID": "P001", "주문일자": "2024-01-15", "수량": 2, "총금액": 64834},
    {"주문ID": "O002", "고객ID": "C002", "상품ID": "P002", "주문일자": "2024-02-20", "수량": 1, "총금액": 25000},
    {"주문ID": "O003", "고객ID": "C003", "상품ID": "P003", "주문일자": "2024-03-05", "수량": 3, "총금액": 107700},
    {"주문ID": "O004", "고객ID": "C001", "상품ID": "P004", "주문일자": "2024-04-10", "수량": 1, "총금액": 15000},
    {"주문ID": "O005", "고객ID": "C002", "상품ID": "P005", "주문일자": "2024-05-12", "수량": 1, "총금액": 120000},
    {"주문ID": "O006", "고객ID": "C003", "상품ID": "P006", "주문일자": "2024-06-18", "수량": 2, "총금액": 40000},
    {"주문ID": "O007", "고객ID": "C001", "상품ID": "P007", "주문일자": "2024-07-22", "수량": 1, "총금액": 45000},
    {"주문ID": "O008", "고객ID": "C002", "상품ID": "P008", "주문일자": "2024-08-30", "수량": 2, "총금액": 36000},
]

# 배송 데이터
deliveries_data = [
    {"배송ID": "D001", "주문ID": "O001", "배송상태": "배송완료", "배송일자": "2024-01-17"},
    {"배송ID": "D002", "주문ID": "O002", "배송상태": "배송중", "배송일자": "2024-02-22"},
    {"배송ID": "D003", "주문ID": "O003", "배송상태": "배송완료", "배송일자": "2024-03-07"},
    {"배송ID": "D004", "주문ID": "O004", "배송상태": "배송완료", "배송일자": "2024-04-12"},
    {"배송ID": "D005", "주문ID": "O005", "배송상태": "배송중", "배송일자": "2024-05-14"},
    {"배송ID": "D006", "주문ID": "O006", "배송상태": "배송완료", "배송일자": "2024-06-20"},
    {"배송ID": "D007", "주문ID": "O007", "배송상태": "배송완료", "배송일자": "2024-07-24"},
    {"배송ID": "D008", "주문ID": "O008", "배송상태": "배송중", "배송일자": None},
]

# 장바구니 데이터
carts_data = [
    {"장바구니ID": " cart001", "고객ID": "C001", "상품ID": "P001", "수량": 2},
    {"장바구니ID": " cart002", "고객ID": "C002", "상품ID": "P002", "수량": 1},
    {"장바구니ID": " cart003", "고객ID": "C003", "상품ID": "P003", "수량": 3},
]

# 리뷰 데이터
reviews_data = [
    {"리뷰ID": "R001", "상품ID": "P001", "고객ID": "C001", "평점": 5, "리뷰내용": "정말 맛있어요! 강아지가 좋아해요.", "작성일자": "2024-01-20"},
    {"리뷰ID": "R002", "상품ID": "P002", "고객ID": "C002", "평점": 4, "리뷰내용": "장난감이 튼튼하고 재미있어요.", "작성일자": "2024-02-25"},
    {"리뷰ID": "R003", "상품ID": "P003", "고객ID": "C003", "평점": 3, "리뷰내용": "스크래처가 조금 작아요.", "작성일자": "2024-03-10"},
    {"리뷰ID": "R004", "상품ID": "P004", "고객ID": "C001", "평점": 5, "리뷰내용": "목줄이 편하고 안전해요.", "작성일자": "2024-04-15"},
    {"리뷰ID": "R005", "상품ID": "P005", "고객ID": "C002", "평점": 4, "리뷰내용": "캣타워가 튼튼하고 고양이가 좋아해요.", "작성일자": "2024-05-18"},
    {"리뷰ID": "R006", "상품ID": "P006", "고객ID": "C003", "평점": 5, "리뷰내용": "샴푸가 부드럽고 향이 좋아요.", "작성일자": "2024-05-18"},
    {"리뷰ID": "R007", "상품ID": "P007", "고객ID": "C001", "평점": 4, "리뷰내용": "사료가 영양가 있고 좋아요.", "작성일자": "2024-07-25"},
]

# 포인트 히스토리 데이터
points_history_data = [
    {"포인트ID": "PT001", "고객ID": "C001", "구분": "적립", "포인트": 100, "내용": "O001 구매 적립", "일자": "2024-01-15"},
    {"포인트ID": "PT002", "고객ID": "C002", "구분": "적립", "포인트": 200, "내용": "O002 구매 적립", "일자": "2024-02-20"},
    {"포인트ID": "PT003", "고객ID": "C003", "구분": "적립", "포인트": 50, "내용": "O003 구매 적립", "일자": "2024-03-05"},
    {"포인트ID": "PT004", "고객ID": "C001", "구분": "적립", "포인트": 150, "내용": "O004 구매 적립", "일자": "2024-04-10"},
    {"포인트ID": "PT005", "고객ID": "C002", "구분": "적립", "포인트": 100, "내용": "O005 구매 적립", "일자": "2024-05-12"},
    {"포인트ID": "PT006", "고객ID": "C003", "구분": "적립", "포인트": 300, "내용": "O006 구매 적립", "일자": "2024-06-05"},
]

# 프로모션 데이터 (현재 진행중인 이벤트나 특가 행사 정보를 담고 있음)
promotions_data = [
    {"프로모션ID": "E001", "제목": "봄맞이 강아지 간식 할인", "내용": "전 상품 10% 할인", "시작일": "2024-03-01", "종료일": "2024-03-31", "대상상품": "전체"},
    {"프로모션ID": "E002", "제목": "고양이 장난감 특가 이벤트", "내용": "특정 장난감 15% 할인", "시작일": "2024-04-01", "종료일": "2024-04-15", "대상상품": "고양이"},
    {"프로모션ID": "E003", "제목": "강아지 목줄 1+1 이벤트", "내용": "강아지 목줄 1+1 제공", "시작일": "2024-05-01", "종료일": "2024-05-10", "대상상품": "목줄"},
    {"프로모션ID": "E004", "제목": "고양이 캣타워 할인 행사", "내용": "고양이 캣타워 25% 할인", "시작일": "2024-06-01", "종료일": "2024-06-30", "대상상품": "캣타워"}
]

# 이렇게 준비한 모든 데이터를 Pandas 데이터프레임으로 반환함.
products_df = pd.DataFrame(products_data)
customers_df = pd.DataFrame(customers_data)
orders_df = pd.DataFrame(orders_data)
deliveries_df = pd.DataFrame(deliveries_data)
carts_df = pd.DataFrame(carts_data)
reviews_df = pd.DataFrame(reviews_data)
points_history_df = pd.DataFrame(points_history_data)
promotions_df = pd.DataFrame(promotions_data)

# 이 데이터들은 서로 ID 를 통해 연결되어 있어서 예를 들어 주문데이터의 고객ID를 통해 고객의 상세 정볼르 조회하는 것이 가능함.

# 에이전트 도구 정의
# 랭체인에서는 @tool 데코레이터를 사용해 일반 파이썬 함수를 에이전트가 호출할 수  있는 도구로 변환할 수 있음.
# 각 도구는 함수의 docstring을 통해 제공하며, 에이전트는 이 설명을 보고 어떤 도구를 사용할 지 판단함.

#@tool
#def calculate_korean_age(birth_year: int) -> int:
#    """사용자의 출생 연도를 입력받아 현재 연도(2026년) 기준의 한국식 나이를 계산합니다.""" #원래는 독스트링""
#    current_year = 2026
#    return current_year - birth_year + 1

# 제대로 변환되었는지 속성 확인해 보기.
# @tool을 붙이는 순간, 이 함수는 단순 계산 기능 뿐만 아니라 LLM에게 전달할 이름{name}과 설명{description} 메타데이터를 갖춘 스마트한 도구로 변환됨.
#print(calculate_korean_age.name) # 출력 : calculate_korean_age
#print(calculate_korean_age.description) # 출력 : 사용자의 출생 연도를 ...


# 먼저 고객 프로필을 조회하는 도구를 만들겠음.
@tool
def get_customer_profile(customer_id: str) -> str:
    """고객 ID로 고객 프로필 정보를 조회합니다."""
    customer = customers_df([customers_df["고객ID"] == customer_id])

    if customer.empty:
        return f"고객 ID {customer_id}를 찾을 수 없습니다."

    # .iloc : Index Location(위치 변경)의 약자임. 내부적으로 부여한 0부터 시작하는 순서를 기준을 데이터를 찾겠다는 의미임 
    c = customer.iloc[0] 
    result = []

    result.append(f"고객ID: {c['고객ID']}")
    result.append(f"고객명: {c['고객명']}")
    result.append(f"이메일: {c['이메일']}")
    result.append(f"전화번호: {c['전화번호']}")
    result.append(f"가입일자: {c['가입일자']}")
    result.append(f"고객등급: {c['고객등급']}")
    result.append(f"보유포인트: {c['보유포인트']}점")
    result.append(f"총구매액: {c['총구매액']}원")
    result.append(f"구매횟수: {c['구매횟수']}회")

    return "\n".join(result)

# 다음으로 상품을 검색하는 도구를 만들겠음.
@tool
def search_products(keyword: str, category: str=None, price_min: int=None, price_max: int=None) -> str:
    """키워드로 상품을 검색합니다."""

    df = products_df.copy()  # 원본 데이터프레임을 복사하여 사용

    # .str.contains(keyword, case=False, na=False) 는 문장 속에서 특정 글자를 찾을 때 쓰는 함수. 
    # keyword : 찾고자 하는 문자열, case: 대소문자 구별 여부, na: 데이터 중에 빈칸(결측지, NaN) 이 있을때 에러를 내지 말고 빈칸은 그냥 False로 처리하겠다는 뜻임.
    if keyword:
        mask = (
            df['상품명'].str.contains(keyword, case=False, na=False) |
            df['태그'].str.contains(keyword, case=False, na=False) |
            df['제조사'].str.contains(keyword, case=False, na=False)
        )
        df = df[mask]

    if category:
        df = df[df['카테고리'].str.contains(category, case=False, na=False)]

    if price_min:
        df = df[df['가격'] >= price_min]

    if price_max:
        df = df[df['가격'] <= price_max]

    if df.empty:
        return "조건에 맞는 상품이 없습니다."

    result = [f"검색 결과: {len(df)}개 상품\n"]

    # Pandas의 데이터프레임의 행을 하나씩 순서대로 꺼내서 처리할 때 사용함.
    # _는 반환된 튜플의 첫번째 요소(행의 인덱스)를 받아오지만 사용하지 않겠다는 의미의 간습적(throwaway) 변수임.
    # _는 그냥 관습일 뿐이고 다른 이름을 써도 되지만, 값이 사용되지 않음을 코드로 명확히 보여주므로 무시할때 흔히 씁니다.
    for _, p in df.iterrows():
        result.append(f"상품ID: {p['상품ID']}")
        result.append(f"상품명: {p['상품명']}")
        result.append(f"가격: {p['가격']}")
        result.append(f"재고: {p['재고']}")
        result.append(f"카테고리: {p['카테고리']}")
        result.append("---------------------")

    return "\n".join(result)

# 고객의 주문내역을 조회하는 도구임.
@tool
def get_customer_orders(customer_id: str, start_date: str=None, end_date: str=None) -> str:
    """고객의 주문내역을 조회합니다. (주문일자 범위 지정 가능)"""
    orders = orders_df[orders_df['고객ID']==customer_id].copy()

    if orders.empty:
        return f"고객 ID {customer_id}의 주문 내역이 없습니다."

    if start_date:
        orders = orders[orders['주문일자'] >= start_date]

    if end_date:
        orders = orders[orders['주문일자'] <= end_date]

    if orders.empty:
        return "해당 기간의 주문 내역이 없습니다."

    result = [f"주문 내역 ({len(orders)}건) : \n"]

    for _, order in orders.iterrows():
        product = products_df[products_df['상품ID']== order['상품ID']]
        product_name = product.iloc[0]['상품명'] if not product.empty else "상품정보없음"

        result.append(f"주문ID: {order['주문ID']}")
        result.append(f"주문일자: {order['주문일자']}")
        result.append(f"상품: {order['상품']}")
        result.append(f"수량: {order['수량']}")
        result.append(f"결제금액: {order['결제금액']}")
        result.append(f"결제상태: {order['결제상태']}")
        result.append("--------------------------")

    return "\n".join(result)

# 배송 상태를 확인하는 도구임.
@tool
def get_delivery_status(customer_id:str=None, order_id:str=None) -> str:
    """배송 상태를 확인합니다."""

    if order_id:
        order = orders_df[orders_df['주문ID'] == order_id]

        if order.empty:
            return f"주문번호 {order_id}를 찾을 수 없습니다."

        delivery_id = order.iloc[0]['배송ID']
        delivery = deliveries_df[deliveries_df['배송ID'] == delivery_id]

        if delivery.empty:
            return "배송 정보를 찾을 수 없습니다."

        d = delivery.iloc[0]

        product = products_df[products_df['상품ID']== order.iloc[0]['상품ID']]
        product_name = product.iloc[0]['상품명'] if not product.empty else '상품정보없음'

        result = []
        result.append(f"주문번호: {order_id}")
        result.append(f"상품: {product_name}")
        result.append(f"배송상태: {d['배송상태']}")

        if d['배송상태'] != "취소":
            result.append(f"배송사:" {d['배송사']})
            result.append(f"송장번호:" {d['송장번호']})
            result.append(f"출고일자:" {d['출고일자']})
            result.append(f"도착예정일:" {d['도착예정일']})

        return "\n".join(result)

    elif customer_id:
        orders = orders_df[orders_df['고객ID']== customer_id]

        if orders.empty:
            return f"고객 ID {customer_id}의 주문이 없습니다."

        result = ["배송 현황:\n"]

        for _, order in orders.iterrows():
            if order['결제상태'] == "결제완료":
                delivery = deliveries_df[deliveries_df['배송ID'] == order['배송ID']]

                if not delivery.empty:
                    d = delivery.iloc[0]
                    product = products_df[products_df['상품ID'] == order['상품ID']]
                    product_name = product.ilic[0]['상품명'] if not product.empty else '상품정보없음'

                    result.append(f"주문번호: {order['주문ID']}")
                    result.append(f"상품: {product_name}")
                    result.append(f"배송상태: {d['배송상태']}")
                    result.append("----------------------")

        return "\n".join(result)

    return "고객ID나 주문ID 를 입력해 주세요"

# 리뷰를 검색하는 도구.
@tool
def search_reviews(keyword: str=None, product_name:str=None, customer_id: str=None, rating: int=None) -> str:
    """리뷰를 검색합니다."""

    df = reviews_df.copy()

    if customer_id:
        df = df[df['고객ID'] == customer_id]

    if product_name:
        matching_products = products_df[
            products_df['상품명'].str.contains(product_name, case=False, na=False)
        ]['상품ID'].toList()

        # .isin() 함수에 원하는 값들을 리스트([])형태로 전달하면, 데이터들을 하나씩 비교하면서 포함돼 있으면 True, 없으면 False 반환.    
        if matching_products:
            df = df[df['상품ID'].isin(matching_products)]

    if keyword:
        df = df[df['리뷰내용'].str.contains(keyword, case=False, na=False)]

    if rating:
        df = df[df['평점'] == rating]

    if df.empty:
        return "조건에 맞는 리뷰가 없습니다."

    result = [f"리뷰 검색 결과 ({len(df)}개): \n"]

    for _, review in df.iterrows():
        product = products_df[products_df['상품ID'] == review['상품ID']]
        product_name = product.iloc[0]['상품명'] if not product.empty else '상품정보없음'

        customer = customers_df[customers_df['고객ID']== review['고객ID']]
        customer_name = customer.iloc[0] if not customer.empty else '고객정보없음'

        result.append(f"[상품]: {product_name}")
        result.append(f"[평점]: {'*' * review['평점']}")
        result.append(f"[리뷰]: {review['리뷰내용']}")
        result.append(f"[작성자]: {customer_name}")
        result.append(f"[작성일]: {review['작성일자']}")
        result.appen('-------------------------')

    return "\n".join(result)

# 장바구니 조회 함수
# 고객의 장바구니에 담긴 상품 목록과 함께 총액, 배송비, 결제 예상액을 계산해서 보여줌.
# 5만원 이상 구매시 배송비가 무료라는 정책이 적용돼 있음.
@tool
def get_customer_cart(customer_id: str)-> str:
    """고객 장바구니를 조회합니다."""
    cart_items = carts_df[carts_df['고객ID']== customer_id]

    if cart_items.empty:
        return f"고객 ID {customer_id}의 장바구니가 비어 있습니다."

    result = [f"장바구니 ({len(cart_items)}개 상품): \n "]

    total = 0

    for _, item in cart_items.iterrows():
        product = products_df[products_df['상품ID']==item['상품iD']]

        if not product.empty:
            p = product.iloc[0]
            subtotal = p['가격'] * item['수량']
            total += subtotal

            result.append(f"상품: {p['상품명']}")
            result.append(f"수량: {item('수량')}개")
            result.append(f"단가: {p['가격']:,}원")
            result.append(f"소계: {subtotal:,}원")
            result.append("-----------------------")

    result.append(f"\n총액: {total:,}원")
    result.append(f"\n배송비: {0 if total > 50000 else 3000:,}원")
    result.append(f"\n결제예상액: {total + (0 if total > 50000 else 3000):,}원")

    return "\n".join(result)

# 포인트를 조회하는 도구
@tool
def get_point_history(customer_id : str) -> str:
    """포인트 내역을 조회합니다."""

    customer = customers_df[customers_df['고객ID'] == customer_id]

    if customer.empty:
        return f"고객 ID {customer_id}를 찾을 수 없습니다."

    current_points = customer.iloc[0]['포인트']
    history = points_history_df[points_history_df['고객ID'] == customer_id]

    result = [f"현재 포인트: {current_points:,}점\n"]

    if not history.empty:
        result.append("포인트 내역:")
        for _, record in history.iterrows():
            result.append(f"{record['일자']} {record['구분']} {record['포인트']:,}점")
            result.append(f"내용: {record['내용']}")
            result.append("-----------------------")

    return "\n".join(result)

# 진행중인 프로모션 조회하는 도구.
@tool
def get_current_promotions() -> str:
    """현재 진행중인 프로모션을 조회합니다."""
    today = datetime.now().strftime("%Y-%m-%d")
    active = promotions_df[
        promotions_df['시작일'] <= today |
        promotions_df['종료일'] >= today
    ]

    if active.empty:
        return "현재 진행중인 프로모션이 없습니다."

    result =["현재 진행 중인 프로모션\n"]

    for _, promo in active.iterrows():
        result.append(f"[{promo['제목']}]")
        result.append(f"내용: {promo['내용']}")
        result.append(f"기간: {promo['시작일']}~{promo['종료일']}")
        result.append("-----------------------")

    return "\n".join(result)

# 인기 상품 조회하는 도구.
@tool
def get_popular_products(category: str = None, period: str=None) -> str:
    """인기 상품을 조회합니다."""

    df = products_df.copy()

    if category:
        df = df[df['카테고리'].str.contains(category, case=False, na=False)]

    # 판매량 기준 정렬.
    df = df.nlargest(5, '판매량')

    if df.empty:
        return "조건에 맞는 상품이 없습니다."

    result = ["인기 상품 Top 5:\n"]

    # enumerate()는 반복 가능한 객체에 인덱스를 붙여 출력할수 있음. enumerate(... , 1) 은 인덱스를 붙이되 1부터 시작하라는 의미임.
    for i, (_, product) in enumerate(df.iterrows(), 1): 
        result.append(f"{i}. {product['상품명']}")
        result.append(f"    가격: {product['가격']:,}원")
        result.append(f"    판매량: {product['판매량']}개")
        result.append("------------------------")

    return "\n".join(result)

IndentationError: unexpected indent (1700619650.py, line 100)

In [ ]:
# 지금까지 정의한 모든 도구를 리스트로 묶어 에이전트에게 전달할 준비를 함.
tools = [
    get_customer_profile,
    search_products,
    get_customer_orders,
    get_delivery_status,
    search_reviews,
    get_customer_cart,
    get_point_history,
    get_current_promotions,
    get_popular_products
]